# 01 — Growth FBA Pipeline

In [1]:
from pathlib import Path
import sys

# Find the project root (parent of notebooks/) so absolute paths work
# from anywhere the notebook is launched.
PROJECT_ROOT = Path('/scratch/ctaylor/core_models_analysis')
REPORTS = PROJECT_ROOT / 'reports'
RESULTS = PROJECT_ROOT / 'results'
LOGS    = PROJECT_ROOT / 'logs'
DATA    = PROJECT_ROOT / 'data'
SCRIPTS = PROJECT_ROOT / 'scripts'
sys.path.insert(0, str(SCRIPTS))

# KBUtils NotebookSession: opens .kbcache/ alongside this notebook so
# heavy outputs can be saved/loaded with provenance.
from kbutillib.notebook import NotebookSession
session = NotebookSession.for_notebook(notebook_file=__file__ if '__file__' in dir() else None,
                                       project_name='core_models_analysis')
print('Cache directory:', session.kbcache_dir)
print('Notebook name :', session.notebook_name)

Cache directory: /scratch/ctaylor/core_models_analysis/notebooks/.kbcache
Notebook name : None


[KBUtilLib] Failed to import rcsb_pdb_utils: ModuleNotFoundError: No module named 'aiohttp'


## What this notebook does

Loads every model in `data/core_models_kegg2/`, restricts exchange
reactions to the ModelSEED `KBaseMedia.cpd` complete media (347
compounds), and runs FBA on `bio1`. A model "grows" if the optimum
biomass flux exceeds 1e-6.

The full multiprocess sweep is implemented in
`scripts/analyze_growth.py`. Here we read the cached `results.csv`
and surface the same numbers `summarize.py` writes into
`SUMMARY.md`.

In [2]:
import csv, pandas as pd
from collections import Counter
from statistics import mean, median

df = pd.read_csv(RESULTS / 'results.csv')
df['grows'] = df['grows'].astype(str).str.lower() == 'true'
print(f'{len(df)} models')
df.head()

5683 models


,model_id,n_metabolites,n_reactions,n_genes,biomass_rxn,n_exchanges_total,n_exchanges_open,status,growth_flux,grows,error
0,GCF_000006985.1,129,125,123,bio1,22,15,optimal,36.464012,True,NaN
1,GCF_000007005.1,136,126,130,bio1,23,16,optimal,0.000000,False,NaN
2,GCF_000007025.1,112,91,74,bio1,19,12,optimal,0.000000,False,NaN
3,GCF_000007045.1,114,106,95,bio1,23,16,optimal,0.000000,False,NaN
4,GCF_000007305.1,101,95,85,bio1,21,14,optimal,0.000000,False,NaN


In [3]:
# Outcome breakdown — same shape as SUMMARY.md
status_counts = Counter(df['status'])
n_grow = int(df['grows'].sum())
n_zero = int(((df['status'] == 'optimal') & (~df['grows'])).sum())
n_other = len(df) - n_grow - n_zero
print(f'  grows (flux > 1e-6):           {n_grow:5d}   ({100*n_grow/len(df):.1f}%)')
print(f'  optimal but zero biomass:      {n_zero:5d}   ({100*n_zero/len(df):.1f}%)')
print(f'  other (infeasible/error/none): {n_other:5d}')
print('  solver-status histogram:', dict(status_counts))

  grows (flux > 1e-6):            3461   (60.9%)
  optimal but zero biomass:       2222   (39.1%)
  other (infeasible/error/none):     0
  solver-status histogram: {'optimal': 5683}


In [4]:
# Cache the grower frame via KBUtils NotebookSession
growers = df[df['grows']].reset_index(drop=True)
session.cache.save('growers_frame', growers, type_hint='dataframe',
                   metadata={'n_growers': len(growers)})
print(f'cached {len(growers)} grower rows under name "growers_frame"')
print(session.cache.load('growers_frame').describe(include='all').iloc[:5])

cached 3461 grower rows under name "growers_frame"
               model_id  n_metabolites  n_reactions      n_genes biomass_rxn  \
count              3461    3461.000000  3461.000000  3461.000000        3461   
unique             3461            NaN          NaN          NaN           1   
top     GCF_000006985.1            NaN          NaN          NaN        bio1   
freq                  1            NaN          NaN          NaN        3461   
mean                NaN     156.608495   171.718867   193.655591         NaN   

        n_exchanges_total  n_exchanges_open   status  growth_flux grows  error  
count         3461.000000       3461.000000     3461  3461.000000  3461    0.0  
unique                NaN               NaN        1          NaN     1    NaN  
top                   NaN               NaN  optimal          NaN  True    NaN  
freq                  NaN               NaN     3461          NaN  3461    NaN  
mean            26.681017         19.144178      NaN    52.3904

In [5]:
# Distributions: biomass flux + reaction counts
flux = growers['growth_flux']
print(f'biomass flux  — min {flux.min():.2f}  median {flux.median():.2f}  '
      f'mean {flux.mean():.2f}  max {flux.max():.2f}')
print(f'n_reactions   — min {df["n_reactions"].min()}  median {df["n_reactions"].median():.0f}  max {df["n_reactions"].max()}')

biomass flux  — min 2.03  median 52.30  mean 52.39  max 87.21
n_reactions   — min 41  median 156  max 221


---
## Report: `reports/SUMMARY.md`

# Core Models KEGG2 — Growth Analysis Summary

- Models analyzed: **5683**
- Media: ModelSEEDDatabase `KBaseMedia.cpd` (complete media, 347 compounds)
- Growth criterion: FBA on biomass reaction (`bio1`) > 1e-6

## Outcomes
| Outcome | Count | % |
|---|---|---|
| **Grows (biomass flux > 1e-6)** | 3461 | 60.9% |
| Optimal solve but zero biomass flux | 2222 | 39.1% |
| Non-optimal solver status (infeasible/unbounded/etc.) | 0 | 0.0% |
| No biomass reaction found | 0 | 0.0% |
| Model load / processing error | 0 | 0.0% |

## Solver status distribution
- `optimal` : 5683

## Biomass flux among growers (n=3461)
- min: 2.03
- median: 52.3
- mean: 52.39
- max: 87.21

## Model sizes (n_reactions)
- min: 41  median: 156  max: 221  mean: 151.4

## Files
- `results/results.csv` — per-model row
- `results/growers.csv` — subset where biomass flux > 1e-6
- `results/non_growers.csv` — everything else (includes errors)
- `logs/failures.log` — solver-status / exception details
- `scripts/analyze_growth.py` — analysis script
- `scripts/summarize.py` — this summary script
- `notebooks/01_GrowthFBA_Pipeline.ipynb` — interactive walkthrough